# (Homework) Week 6 - DataScience Bootcamp Fall 2025

All solution cells are replaced with `# TODO` placeholders so you can fill them in.

**Name:** \
**Email:**

---

### Problem 1: Dataset Splitting

1. You have recordings of 44 phones from 100 people; each person records ~200 phones/day for 5 days.
   - Design a valid training/validation/test split strategy that ensures the model generalizes to **new speakers**.

2. You now receive an additional dataset of 10,000 phone recordings from **Kilian**, a single speaker.
   - You must train a model that performs well **specifically for Kilian**, while also maintaining generalization.

*Describe your proposed split strategy and reasoning.* (Theory)

In [1]:
#Todo
"""
I use a speaker-independent split for the original 100-speaker dataset: 70% of speakers for training, 15% for validation, and 15% for testing. 
This ensures that no speaker appears in more than one split, forcing the model to generalize to completely new voices.

Kilian’s 10,000-recording dataset is treated as a separate domain. I first train a general model using only the multi-speaker data. Then I split Kilian’s data into 80% train, 10% validation, and 10% test, 
and perform light fine-tuning on the Kilian train set while monitoring the dedicated Kilian validation set. This setup allows the model to specialize on Kilian without overfitting or losing generalization to new speakers.

"""

'\nI use a speaker-independent split for the original 100-speaker dataset: 70% of speakers for training, 15% for validation, and 15% for testing. \nThis ensures that no speaker appears in more than one split, forcing the model to generalize to completely new voices.\n\nKilian’s 10,000-recording dataset is treated as a separate domain. I first train a general model using only the multi-speaker data. Then I split Kilian’s data into 80% train, 10% validation, and 10% test, \nand perform light fine-tuning on the Kilian train set while monitoring the dedicated Kilian validation set. This setup allows the model to specialize on Kilian without overfitting or losing generalization to new speakers.\n\n'

### Problem 2: K-Nearest Neighbors

1. **1-NN Classification:** Given dataset:

   Positive: (1,2), (1,4), (5,4)

   Negative: (3,1), (3,2)

   Plot the 1-NN decision boundary and classify new points visually.

2. **Feature Scaling:** Consider dataset:

   Positive: (100,2), (100,4), (500,4)

   Negative: (300,1), (300,2)

   What would the 1-NN classify point (500,1) as **before and after scaling** to [0,1] per feature?

3. **Handling Missing Values:** How can you modify K-NN to handle missing features in a test point?

4. **High-dimensional Data:** Why can K-NN still work well for images even with thousands of pixels?


In [ ]:
#Todo

# 1.
import numpy as np
import matplotlib.pyplot as plt

# Given data
positive = np.array([[1,2],[1,4],[5,4]])
negative = np.array([[3,1],[3,2]])

# Plot points
plt.scatter(positive[:,0], positive[:,1], color='green', label='Positive')
plt.scatter(negative[:,0], negative[:,1], color='red', label='Negative')

# Optional: plot a grid of predictions
xs = np.linspace(0,6,200)
ys = np.linspace(0,6,200)
grid = np.array([(x,y) for x in xs for y in ys])

def classify(point):
    points = np.vstack((positive, negative))
    labels = np.array([1,1,1,0,0])   # 1 = positive, 0 = negative
    dists = np.linalg.norm(points - point, axis=1)
    return labels[np.argmin(dists)]

preds = np.array([classify(p) for p in grid])
colors = ['lightgreen' if p==1 else 'mistyrose' for p in preds]

plt.scatter(grid[:,0], grid[:,1], c=colors, s=3, alpha=0.3)
plt.legend()
plt.title("1-NN Classification Boundary")
plt.show()

# 2. 
import numpy as np

# Dataset
positive = np.array([[100,2],[100,4],[500,4]])
negative = np.array([[300,1],[300,2]])
query = np.array([500,1])

# === BEFORE SCALING ===
def nearest_before(query):
    points = np.vstack((positive, negative))
    labels = np.array([1,1,1,0,0])
    d = np.linalg.norm(points - query, axis=1)
    return labels[np.argmin(d)], d

label_before, d_before = nearest_before(query)
label_before
# === AFTER SCALING TO [0,1] ===

# scale function
def minmax_scale(X):
    mins = X.min(axis=0)
    maxs = X.max(axis=0)
    return (X - mins) / (maxs - mins), mins, maxs

all_points = np.vstack((positive, negative, query))
scaled, mins, maxs = minmax_scale(all_points)

# separate back out
pos_s = scaled[:3]
neg_s = scaled[3:5]
qry_s = scaled[-1]

# classify after scaling
points = np.vstack((pos_s, neg_s))
labels = np.array([1,1,1,0,0])
dist = np.linalg.norm(points - qry_s, axis=1)
label_after = labels[np.argmin(dist)]

label_after

# 3. Ignore the missing dimensions when computing distances, or impute missing values using statistics or K-NN–based imputation.

# 4. Because real images lie on structured low-dimensional manifolds, and similar images remain close in pixel-space, allowing K-NN to find meaningful neighbors despite the high number of raw features.


### Problem 3: Part 1

You are given a fully trained Perceptron model with weight vector **w**, along with training set **D_TR** and test set **D_TE**.

1. Your co-worker suggests evaluating $h(x) = sign(w \cdot x)$ for every $(x, y)$ in D_TR and D_TE. Does this help determine whether test error is higher than training error?
2. Why is there no need to compute training error explicitly for the Perceptron algorithm?

In [ ]:
#Todo
"""
Q1.
Yes — evaluating on both datasets gives the predictions for every example, and from these predictions 
you can directly count the number of mistakes on the training set and on the test set. Once you know 
how many training mistakes vs. test mistakes occur, you can compare the two and determine whether test 
error is higher than training error.

Q2. 
There is no need to compute the training error because the Perceptron algorithm continues updating its 
weight vector until it correctly classifies all training examples (assuming the data is linearly separable). 
The algorithm only stops once training error is zero. Thus, after training finishes, you automatically know the training error without needing to compute it.
"""

### Problem 3: Two-point 2D Dataset (Part 2)

Run the Perceptron algorithm **by hand or in code** on the following data:

1. Positive class: (10, -2)
2. Negative class: (12, 2)

Start with $w_0 = (0, 0)$ and a learning rate of 1.

- Compute how many updates are required until convergence.
- Write down the sequence of $w_i$ vectors.

In [ ]:
# Todo

# 1
import numpy as np

# Dataset
X = np.array([
    [10, -2],   # positive
    [12,  2]    # negative
])
y = np.array([+1, -1])   # labels

# Initialization
w = np.array([0.0, 0.0])
lr = 1
updates = 0

weights_sequence = [w.copy()]

converged = False

while not converged:
    converged = True
    for xi, yi in zip(X, y):
        if yi * np.dot(w, xi) <= 0:  
            # misclassified → update
            w = w + lr * yi * xi
            updates += 1
            weights_sequence.append(w.copy())
            converged = False  # need another pass

updates, weights_sequence


### Problem 4: Reconstructing the Weight Vector

Given the log of Perceptron updates:

| x | y | count |
|---|---|--------|
| (0, 0, 0, 0, 4) | +1 | 2 |
| (0, 0, 6, 5, 0) | +1 | 1 |
| (3, 0, 0, 0, 0) | -1 | 1 |
| (0, 9, 3, 6, 0) | -1 | 1 |
| (0, 1, 0, 2, 5) | -1 | 1 |

Assume learning rate = 1 and initial weight $w_0 = (0, 0, 0, 0, 0)$.

Compute the final weight vector after all updates.

In [ ]:
#Todo
(-4, -9, 1, -3, 3)

### Problem 5: Visualizing Perceptron Convergence

Implement a Perceptron on a small 2D dataset with positive and negative examples.

- Plot the data points.
- After each update, visualize the decision boundary.
- Show how it converges to a stable separator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Create a small linearly separable 2D dataset
X = np.array([
    [1, 1],
    [2, 1],
    [1, 2],   # Positive class
    [4, 4],
    [5, 4],
    [4, 5]    # Negative class
])
y = np.array([+1, +1, +1, -1, -1, -1])

# Add bias term (x1, x2, 1)
Xb = np.hstack([X, np.ones((X.shape[0], 1))])


# 2. Helper function to plot the current decision boundary
def plot_boundary(w, X, y, title):
    plt.figure(figsize=(5,5))

    # Plot data points
    plt.scatter(X[y==1][:,0], X[y==1][:,1], c='green', label='Positive', s=60)
    plt.scatter(X[y==-1][:,0], X[y==-1][:,1], c='red', label='Negative', s=60)

    # Plot decision boundary w1*x + w2*y + w3 = 0
    if w[1] != 0:
        xs = np.linspace(0, 6, 200)
        ys = -(w[0] * xs + w[2]) / w[1]
        plt.plot(xs, ys, 'b-', linewidth=2, label='Decision boundary')

    plt.xlim(0, 6)
    plt.ylim(0, 6)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


# 3. Perceptron Training Loop (with visualization)
w = np.zeros(3, dtype=float)   # weights: w1, w2, bias
lr = 1
updates = 0

# Plot initial boundary
plot_boundary(w, Xb, y, f"Initial boundary (w={w})")

converged = False
iteration = 1

while not converged and updates < 50:  # safety cap
    converged = True
    for xi, yi in zip(Xb, y):
        if yi * np.dot(w, xi) <= 0:  # misclassified
            w = w + lr * yi * xi
            updates += 1
            converged = False
            plot_boundary(w, Xb, y, f"Update {updates}: w = {w}")
    iteration += 1

print("Final weight vector:", w)
print("Total updates:", updates)
